In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "data/AAPL.csv",
    skiprows=2,
    names=["Date", "Close", "High", "Low", "Open", "Volume"]
)

# 핵심: 파싱 실패는 NaT로
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Date가 NaT인 행(찌꺼기) 제거
df = df.dropna(subset=["Date"]).copy()

# 숫자 컬럼도 안전하게 변환 (문자 찌꺼기 있으면 NaN)
for c in ["Close", "High", "Low", "Open", "Volume"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna().copy()

df = df.sort_values("Date").reset_index(drop=True)

print(df.head(3))
print(df.tail(3))
print(df.dtypes)
print("rows:", len(df))


        Date      Close       High        Low       Open       Volume
0 2015-12-21  24.199585  24.208604  23.802759  24.188311  190362400.0
1 2015-12-22  24.177036  24.287516  24.001169  24.215366  131157600.0
2 2015-12-23  24.488176  24.542288  24.170264  24.186047  130629600.0
           Date       Close        High         Low        Open       Volume
2512 2025-12-17  271.839996  276.160004  271.640015  275.010010   50138700.0
2513 2025-12-18  272.190002  273.630005  266.950012  273.609985   51630700.0
2514 2025-12-19  273.670013  274.600006  269.899994  272.149994  144632000.0
Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume           float64
dtype: object
rows: 2515


/tmp/ipykernel_24079/732093123.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


#### 타깃 만들기 (내일 상승/하락)

In [2]:
df["y_up"] = (df["Close"].shift(-1) > df["Close"]).astype(int)
df = df.dropna(subset=["y_up"]).copy()

print("rows(after y):", len(df))
print("y 비율(상승=1):", df["y_up"].mean())

rows(after y): 2515
y 비율(상승=1): 0.5359840954274354


#### 시계열 분할

In [3]:
train_end = "2022-12-31"
val_end   = "2024-06-30"

train = df[df["Date"] <= train_end].copy()
val   = df[(df["Date"] > train_end) & (df["Date"] <= val_end)].copy()
test  = df[df["Date"] > val_end].copy()

print("train/val/test:", len(train), len(val), len(test))
print("train y mean:", train["y_up"].mean())
print("val y mean:", val["y_up"].mean())
print("test y mean:", test["y_up"].mean())


train/val/test: 1770 374 371
train y mean: 0.5299435028248588
val y mean: 0.5454545454545454
test y mean: 0.555256064690027


#### Baseline 피쳐 만들기

In [4]:
def make_features(df):
    out = df.copy()

    # 1) 로그 수익률 (가장 중요)
    out["log_ret"] = np.log(out["Close"] / out["Close"].shift(1))

    # 2) 변동성 proxy
    out["hl_spread"] = (out["High"] - out["Low"]) / out["Close"]

    # 3) 시가-종가 방향성
    out["oc_spread"] = (out["Close"] - out["Open"]) / out["Open"]

    # 4) 거래량 변화
    out["vol_chg"] = np.log(out["Volume"] / out["Volume"].shift(1))

    # 5) 단기 모멘텀
    out["ret_5"] = out["log_ret"].rolling(5).sum()

    # 6) 중기 모멘텀
    out["ret_20"] = out["log_ret"].rolling(20).sum()

    return out


In [5]:
train_f = make_features(train).dropna().copy()
val_f   = make_features(val).dropna().copy()
test_f  = make_features(test).dropna().copy()

features = [
    "log_ret", "hl_spread", "oc_spread",
    "vol_chg", "ret_5", "ret_20"
]

X_train = train_f[features]
y_train = train_f["y_up"]

X_val = val_f[features]
y_val = val_f["y_up"]

X_test = test_f[features]
y_test = test_f["y_up"]


#### Baseline Logistic Regression

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

pipe.fit(X_train, y_train)

for name, X, y in [
    ("train", X_train, y_train),
    ("val", X_val, y_val),
    ("test", X_test, y_test),
]:
    pred = pipe.predict(X)
    acc = accuracy_score(y, pred)
    print(f"{name} acc: {acc:.4f}")


train acc: 0.5326
val acc: 0.5311
test acc: 0.5071


#### 계수 보기

In [8]:
coef = pipe.named_steps["clf"].coef_[0]
for f, c in sorted(zip(features, coef), key=lambda x: abs(x[1]), reverse=True):
    print(f"{f:12s} {c:+.4f}")


log_ret      -0.1377
ret_5        +0.0403
vol_chg      -0.0347
ret_20       +0.0266
hl_spread    -0.0245
oc_spread    +0.0164



log_ret (전일 수익률) 음수

어제 많이 올랐으면 → 내일은 떨어질 확률 ↑

-> 단기 mean-reversion 신호

ret_5, ret_20 양수

최근 며칠~몇 주 상승 추세면 → 계속 오를 확률 ↑

-> 중기 momentum 신호

vol_chg 음수

거래량 급증 → 불확실성/분산 ↑ → 방향성 약화

이건 전형적인 금융 직관과 완전히 일치한다.
즉, 모델이 약한 게 아닌 신호가 약한 시장이다.

#### Confusion matrix 보기

In [9]:
from sklearn.metrics import confusion_matrix

pred_test = pipe.predict(X_test)
print(confusion_matrix(y_test, pred_test))

[[ 13 146]
 [ 27 165]]


#### 확률 기반 평가

In [10]:
proba = pipe.predict_proba(X_test)[:, 1]

# 상위 20% 확률만 매수한다고 가정
threshold = np.quantile(proba, 0.8)
signal = (proba >= threshold).astype(int)

print(confusion_matrix(y_test, signal))
print("매수 비율:", signal.mean())


[[122  37]
 [158  34]]
매수 비율: 0.2022792022792023


#### 기대수익 관점 평가

In [11]:
test_f = test_f.iloc[-len(proba):].copy()
test_f["signal"] = signal
test_f["ret_next"] = np.log(
    test_f["Close"].shift(-1) / test_f["Close"]
)

print("전략 평균 수익:", test_f.loc[test_f["signal"]==1, "ret_next"].mean())
print("전체 평균 수익:", test_f["ret_next"].mean())


전략 평균 수익: -0.0013220036468221267
전체 평균 수익: 0.0006586683129647033


#### XGBoost 모델 준비

In [12]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
)

#### 학습

In [15]:
xgb.fit(X_train, y_train)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


#### Accuracy 비교

In [16]:
for name, X, y in [
    ("train", X_train, y_train),
    ("val", X_val, y_val),
    ("test", X_test, y_test),
]:
    pred = xgb.predict(X)
    acc = accuracy_score(y, pred)
    print(f"{name} acc: {acc:.4f}")


train acc: 0.7674
val acc: 0.5593
test acc: 0.5157


#### 평가 : 확률기반 + 기대수익

In [17]:
proba_xgb = xgb.predict_proba(X_test)[:, 1]

threshold = np.quantile(proba_xgb, 0.8)
signal_xgb = (proba_xgb >= threshold).astype(int)

print(confusion_matrix(y_test, signal_xgb))
print("매수 비율:", signal_xgb.mean())

[[130  29]
 [150  42]]
매수 비율: 0.2022792022792023


In [18]:
test_fx = test_f.iloc[-len(proba_xgb):].copy()
test_fx["signal"] = signal_xgb
test_fx["ret_next"] = np.log(
    test_fx["Close"].shift(-1) / test_fx["Close"]
)

print("XGB 전략 평균 수익:",
      test_fx.loc[test_fx["signal"]==1, "ret_next"].mean())
print("전체 평균 수익:", test_fx["ret_next"].mean())

XGB 전략 평균 수익: 0.004017447089082051
전체 평균 수익: 0.0006586683129647033


#### 윈도우용 데이터셋 준비

In [19]:
import numpy as np

def make_windows(df_feat, feature_cols, target_col="y_up", window=20):
    X = df_feat[feature_cols].to_numpy()
    y = df_feat[target_col].to_numpy()

    Xs, ys = [], []
    for i in range(window - 1, len(df_feat)):
        Xs.append(X[i-window+1:i+1])   # 과거 window일 포함
        ys.append(y[i])                # 그 시점의 타깃(내일 상승/하락)
    return np.array(Xs), np.array(ys)

In [20]:
window = 20  # 일단 20일로 시작 (한 달 거래일쯤)
Xtr_w, ytr_w = make_windows(train_f, features, window=window)
Xva_w, yva_w = make_windows(val_f, features, window=window)
Xte_w, yte_w = make_windows(test_f, features, window=window)

print("Xtr_w:", Xtr_w.shape, "ytr_w:", ytr_w.shape)
print("Xva_w:", Xva_w.shape, "yva_w:", yva_w.shape)
print("Xte_w:", Xte_w.shape, "yte_w:", yte_w.shape)


Xtr_w: (1731, 20, 6) ytr_w: (1731,)
Xva_w: (335, 20, 6) yva_w: (335,)
Xte_w: (332, 20, 6) yte_w: (332,)


#### CNN 모델 정의

In [21]:
import torch
import torch.nn as nn

class CNN1D(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.conv1 = nn.Conv1d(
            in_channels=n_features,
            out_channels=32,
            kernel_size=3,
            padding=1
        )
        self.conv2 = nn.Conv1d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        # x: (batch, window, features)
        x = x.transpose(1, 2)   # (batch, features, window)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)
        x = self.fc(x).squeeze(-1)
        return x


#### Tensor 변환

In [22]:
Xtr_t = torch.tensor(Xtr_w, dtype=torch.float32)
ytr_t = torch.tensor(ytr_w, dtype=torch.float32)

Xva_t = torch.tensor(Xva_w, dtype=torch.float32)
yva_t = torch.tensor(yva_w, dtype=torch.float32)

Xte_t = torch.tensor(Xte_w, dtype=torch.float32)
yte_t = torch.tensor(yte_w, dtype=torch.float32)


#### 학습 루프

In [23]:
model = CNN1D(n_features=6)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(20):
    model.train()
    optimizer.zero_grad()

    logits = model(Xtr_t)
    loss = criterion(logits, ytr_t)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(Xva_t)
        val_loss = criterion(val_logits, yva_t)

    if epoch % 5 == 0:
        print(f"epoch {epoch:02d} | train loss {loss:.4f} | val loss {val_loss:.4f}")


epoch 00 | train loss 0.6912 | val loss 0.6904
epoch 05 | train loss 0.6911 | val loss 0.6904
epoch 10 | train loss 0.6910 | val loss 0.6903
epoch 15 | train loss 0.6910 | val loss 0.6903


#### 확률 기반 + 기대수익 평가 (XGB와 동일 조건)

In [24]:
model.eval()
with torch.no_grad():
    proba_cnn = torch.sigmoid(model(Xte_t)).numpy()

threshold = np.quantile(proba_cnn, 0.8)
signal_cnn = (proba_cnn >= threshold).astype(int)

print("매수 비율:", signal_cnn.mean())
print(confusion_matrix(yte_w, signal_cnn))


매수 비율: 0.20180722891566266
[[121  32]
 [144  35]]


In [25]:
test_cnn = test_f.iloc[-len(proba_cnn):].copy()
test_cnn["signal"] = signal_cnn
test_cnn["ret_next"] = np.log(
    test_cnn["Close"].shift(-1) / test_cnn["Close"]
)

print("CNN 전략 평균 수익:",
      test_cnn.loc[test_cnn["signal"]==1, "ret_next"].mean())
print("전체 평균 수익:", test_cnn["ret_next"].mean())


CNN 전략 평균 수익: -0.0007620571206973978
전체 평균 수익: 0.0005794337099703419


CNN은 아무 패턴도 못 배웠다

#### Feature Importance 3종 뽑기

In [27]:
import pandas as pd

# 1) 기본 sklearn 방식 (gain 기반일 수도 있고 버전에 따라 다름)
imp_sklearn = pd.Series(xgb.feature_importances_, index=features).sort_values(ascending=False)
print("== sklearn feature_importances_ ==")
print(imp_sklearn)

# 2) XGBoost booster 기반 importance: weight / gain / cover
# booster 기반 importance (버전 안전)
booster = xgb.get_booster()

for t in ["weight", "gain", "cover"]:
    score = booster.get_score(importance_type=t)

    # key가 이미 feature name인 경우 그대로 사용
    s = pd.Series(score).sort_values(ascending=False)

    print(f"\n== importance_type='{t}' ==")
    print(s)
    


== sklearn feature_importances_ ==
log_ret      0.173720
vol_chg      0.170174
hl_spread    0.169314
oc_spread    0.166238
ret_20       0.162436
ret_5        0.158119
dtype: float32

== importance_type='weight' ==
vol_chg      365.0
log_ret      333.0
ret_20       324.0
oc_spread    302.0
hl_spread    298.0
ret_5        278.0
dtype: float64

== importance_type='gain' ==
log_ret      3.000124
vol_chg      2.938880
hl_spread    2.924028
oc_spread    2.870910
ret_20       2.805241
ret_5        2.730694
dtype: float64

== importance_type='cover' ==
hl_spread    170.854675
ret_20       166.554276
vol_chg      162.673401
oc_spread    160.626358
log_ret      157.564117
ret_5        139.043549
dtype: float64


#### 조건 추출

In [28]:
import numpy as np

# val에서 확률 뽑기 (test는 아껴두자)
proba_val = xgb.predict_proba(X_val)[:, 1]
thr = np.quantile(proba_val, 0.8)
sig = proba_val >= thr

summary = []
for f in features:
    summary.append({
        "feature": f,
        "mean_all": float(X_val[f].mean()),
        "mean_signal(top20%)": float(X_val.loc[sig, f].mean()),
        "diff(signal-all)": float(X_val.loc[sig, f].mean() - X_val[f].mean())
    })

df_sum = pd.DataFrame(summary).sort_values("diff(signal-all)", ascending=False)
print(df_sum)


     feature  mean_all  mean_signal(top20%)  diff(signal-all)
1  hl_spread  0.016668             0.017720          0.001052
5     ret_20  0.023909             0.024078          0.000169
2  oc_spread  0.001149            -0.000835         -0.001985
0    log_ret  0.001092            -0.000989         -0.002080
4      ret_5  0.005533             0.002090         -0.003443
3    vol_chg  0.000637            -0.053529         -0.054166


#### 재현

In [29]:
# 기준선: val 분포로 컷 설정
thr_ret20 = X_val["ret_20"].median()
thr_hl    = X_val["hl_spread"].median()

rule_signal = (
    (X_test["ret_20"] > thr_ret20) &
    (X_test["hl_spread"] > thr_hl)
).astype(int)

print("룰 매수 비율:", rule_signal.mean())
print(confusion_matrix(y_test, rule_signal))


룰 매수 비율: 0.31339031339031337
[[102  57]
 [139  53]]


In [30]:
test_rule = test_f.iloc[-len(rule_signal):].copy()
test_rule["signal"] = rule_signal
test_rule["ret_next"] = np.log(
    test_rule["Close"].shift(-1) / test_rule["Close"]
)

print("룰 전략 평균 수익:",
      test_rule.loc[test_rule["signal"]==1, "ret_next"].mean())
print("전체 평균 수익:", test_rule["ret_next"].mean())


룰 전략 평균 수익: 0.0007910509678052056
전체 평균 수익: 0.0006586683129647033


#### 하이브리드 신호 생성

In [31]:
import numpy as np

# 1) 룰 threshold (val 기준)
thr_ret20 = X_val["ret_20"].median()
thr_hl    = X_val["hl_spread"].median()

# 2) test 룰 필터
rule_test = (
    (X_test["ret_20"] > thr_ret20) &
    (X_test["hl_spread"] > thr_hl)
)

# 3) XGB 확률
proba_test = xgb.predict_proba(X_test)[:, 1]

# 4) XGB 내부 threshold (상위 20%)
thr_proba = np.quantile(proba_test[rule_test], 0.8)

# 5) 포지션 크기 (0~1)
position = np.zeros(len(proba_test))
position[rule_test] = np.clip(
    (proba_test[rule_test] - thr_proba) / (1 - thr_proba),
    0, 1
)

print("평균 포지션 크기:", position.mean())
print("룰 통과 비율:", rule_test.mean())


평균 포지션 크기: 0.009419894838265204
룰 통과 비율: 0.31339031339031337


#### 기대수익 평가

In [32]:
test_hybrid = test_f.iloc[-len(position):].copy()
test_hybrid["position"] = position
test_hybrid["ret_next"] = np.log(
    test_hybrid["Close"].shift(-1) / test_hybrid["Close"]
)

# 포지션 반영 수익
test_hybrid["strategy_ret"] = (
    test_hybrid["position"] * test_hybrid["ret_next"]
)

print("하이브리드 전략 평균 수익:",
      test_hybrid["strategy_ret"].mean())

print("전체 평균 수익:",
      test_hybrid["ret_next"].mean())


하이브리드 전략 평균 수익: 4.78844487981498e-05
전체 평균 수익: 0.0006586683129647033


#### 리스크 감각 점검

In [33]:
print("전략 표준편차:",
      test_hybrid["strategy_ret"].std())

print("Sharpe (일단 무위험 0 가정):",
      test_hybrid["strategy_ret"].mean() /
      test_hybrid["strategy_ret"].std())

전략 표준편차: 0.0006738129425583281
Sharpe (일단 무위험 0 가정): 0.07106489913408678


#### threshold 조정 및 포지션 스케일 재설계

In [34]:
thr_proba = 0.6

position = np.zeros(len(proba_test))
position[rule_test] = np.clip(
    (proba_test[rule_test] - thr_proba) / (0.9 - thr_proba),
    0, 1
)

test_hybrid["position"] = position
test_hybrid["strategy_ret"] = (
    test_hybrid["position"] * test_hybrid["ret_next"]
)

print("평균 포지션:", position.mean())
print("전략 평균 수익:", test_hybrid["strategy_ret"].mean())
print("Sharpe:",
      test_hybrid["strategy_ret"].mean() /
      test_hybrid["strategy_ret"].std())


평균 포지션: 0.017814836584222622
전략 평균 수익: 9.019005666988344e-05
Sharpe: 0.08130160981713287


#### 개선 평가 코드 (Power Scaling 포지션)

In [36]:
# Power scaling 파라미터
gamma = 2.0   # 1.5, 2.0, 3.0 중 하나부터

position = np.zeros(len(proba_test))
position[rule_test] = np.clip(
    ((proba_test[rule_test] - 0.5) / 0.5) ** gamma,
    0, 1
)

test_hybrid["position"] = position
test_hybrid["strategy_ret"] = (
    test_hybrid["position"] * test_hybrid["ret_next"]
)

print("gamma:", gamma)
print("평균 포지션:", position.mean())
print("전략 평균 수익:", test_hybrid["strategy_ret"].mean())
print("Sharpe:",
      test_hybrid["strategy_ret"].mean() /
      test_hybrid["strategy_ret"].std())


gamma: 2.0
평균 포지션: 0.01504316019290958
전략 평균 수익: 5.415467255246328e-05
Sharpe: 0.08721303050333755


- volatilty scaling

#### 실현 변동성 계산 (20일)

In [37]:
# 20일 실현 변동성
test_hybrid["vol_20"] = (
    test_hybrid["ret_next"]
    .rolling(20)
    .std()
)


#### 변동성 타겟 포지션

In [38]:
target_vol = 0.01  # 하루 1% 변동성 타겟

position_vol = np.zeros(len(test_hybrid))
mask = rule_test.values  # 룰 통과한 날만

position_vol[mask] = np.clip(
    target_vol / test_hybrid.loc[mask, "vol_20"],
    0, 1
)

test_hybrid["position"] = position_vol
test_hybrid["strategy_ret"] = (
    test_hybrid["position"] * test_hybrid["ret_next"]
)


#### 최종 성적

In [39]:
print("평균 포지션:", test_hybrid["position"].mean())
print("전략 평균 수익:", test_hybrid["strategy_ret"].mean())
print("Sharpe:",
      test_hybrid["strategy_ret"].mean() /
      test_hybrid["strategy_ret"].std())


평균 포지션: 0.21984103938131752
전략 평균 수익: 0.00012753302061188575
Sharpe: 0.02196668745799872
